# RetailStream Inc. — Batch, Incremental & Streaming Pipeline

**Author:** Nidhi Bansal &nbsp;|&nbsp; **Track:** Data Engineering (PySpark / SparkSQL / Delta Lake) &nbsp;

RetailStream notebook builds the **Bronze → Silver → Gold** pipeline described in the brief : 
an initial
batch load, an incremental append with dedup, a streaming Auto Loader ingestion for payments, a late-arriving
file handled with `MERGE`, an enriched Silver table, and two Gold aggregates queried with SparkSQL.


###  Setup — paths, schemas

In [0]:

dbutils.widgets.text("base_path", "/Volumes/databricksmaster/default/retailstream", "Base DBFS path")
BASE = dbutils.widgets.get("base_path")
DATA        = f"{BASE}/data"
CHECKPOINTS = f"{BASE}/checkpoints"
BRONZE      = f"{BASE}/delta/bronze"
SILVER      = f"{BASE}/delta/silver"
GOLD        = f"{BASE}/delta/gold"

BRONZE_ORDERS_PATH        = f"{BRONZE}/bronze_orders"
BRONZE_TXN_PATH           = f"{BRONZE}/bronze_transactions"
SILVER_ORDERS_PATH        = f"{SILVER}/silver_orders"
GOLD_MONTHLY_SALES_PATH   = f"{GOLD}/gold_monthly_sales"
GOLD_PAYMENT_SUMMARY_PATH = f"{GOLD}/gold_payment_summary"

AUTOLOADER_LANDING    = f"{DATA}/autoloader_landing"
AUTOLOADER_CHECKPOINT = f"{CHECKPOINTS}/bronze_transactions"
AUTOLOADER_SCHEMA_LOC = f"{CHECKPOINTS}/bronze_transactions_schema"

print("Base path:", BASE)

Base path: /retailstream


In [0]:
orders_schema = StructType([
    StructField("order_id",    StringType(),  False),
    StructField("customer_id", StringType(),  True),
    StructField("product_id",  StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("order_date",  DateType(),    True),
    StructField("store_id",    StringType(),  True),
    StructField("status",      StringType(),  True),
])

transactions_schema = StructType([
    StructField("txn_id",         StringType(),    False),
    StructField("order_id",       StringType(),    True),
    StructField("payment_method", StringType(),    True),
    StructField("amount",         DoubleType(),    True),
    StructField("txn_timestamp",  TimestampType(), True),
    StructField("currency",       StringType(),    True),
    StructField("gateway_status", StringType(),    True),
])

**Dimension tables (products, customers, stores)** are smaller, static reference datasets. I used inferSchema=True here as they don't have the same risk of data-type drift as our high-volume fact files. For the fact files, I’m sticking to an explicit StructType schema to ensure strictly validated data ingestion.

In [0]:
DIM_DATA = "/Volumes/databricksmaster/default/retailstream/data"

products_df  = spark.read.csv(f"{DIM_DATA}/products.csv",  header=True, inferSchema=True)
customers_df = spark.read.csv(f"{DIM_DATA}/customers.csv", header=True, inferSchema=True)
stores_df    = spark.read.csv(f"{DIM_DATA}/stores.csv",    header=True, inferSchema=True)

print("products:", products_df.count(), "| customers:", customers_df.count(), "| stores:", stores_df.count())
stores_df.show()

products: 8 | customers: 33 | stores: 4
+--------+--------------+------+---------+
|store_id|    store_name|region|     city|
+--------+--------------+------+---------+
|     S01|Mumbai Central|  West|   Mumbai|
|     S02| Bangalore Hub| South|Bangalore|
|     S03|     Delhi NCR| North|    Delhi|
|     S04|Ahmedabad West|  West|Ahmedabad|
+--------+--------------+------+---------+



###Task 1 — Initial Batch Load (Bronze) 
 I'm loading the January batch and using an explicit schema instead of inferSchema.
 It’s safer this way—if the data types change, the pipeline fails early rather 
 than breaking downstream. Adding audit columns (ingested_at/source_file) is
 standard for tracking data lineage and helps me debug if a specific file is corrupted.

In [0]:
DATA = "/Volumes/databricksmaster/default/retailstream/data"
BRONZE_ORDERS_PATH = "/Volumes/databricksmaster/default/retailstream/delta/bronze/bronze_orders"
jan_file = "orders_2024_01.csv"

jan_df = (
    spark.read.csv(f"{DATA}/batch_initial/{jan_file}", header=True, schema=orders_schema)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", lit(jan_file))
)

(jan_df.write
    .format("delta")
    .mode("overwrite")
    .save(BRONZE_ORDERS_PATH))

bronze_orders = spark.read.format("delta").load(BRONZE_ORDERS_PATH)
jan_count = bronze_orders.count()
print(f"bronze_orders after initial load: {jan_count} rows")
assert jan_count == 20, f"expected 20 rows after Task 1, got {jan_count}"

bronze_orders.orderBy("order_id").show(5, truncate=False)

bronze_orders after initial load: 20 rows
+--------+-----------+----------+--------+----------+----------+--------+---------+-------------------------+------------------+
|order_id|customer_id|product_id|quantity|unit_price|order_date|store_id|status   |ingested_at              |source_file       |
+--------+-----------+----------+--------+----------+----------+--------+---------+-------------------------+------------------+
|ORD001  |C026       |P008      |1       |28000.0   |2024-01-11|S02     |DELIVERED|2026-07-11 20:54:54.50394|orders_2024_01.csv|
|ORD002  |C021       |P004      |4       |18000.0   |2024-01-12|S03     |DELIVERED|2026-07-11 20:54:54.50394|orders_2024_01.csv|
|ORD003  |C020       |P003      |2       |50000.0   |2024-01-04|S01     |DELIVERED|2026-07-11 20:54:54.50394|orders_2024_01.csv|
|ORD004  |C031       |P007      |1       |18000.0   |2024-01-09|S03     |DELIVERED|2026-07-11 20:54:54.50394|orders_2024_01.csv|
|ORD005  |C005       |P004      |3       |800.0     |20

###Task 2 — Incremental Batch Load, Append Only (No Duplicates)
 I noticed the February file actually contains some duplicate order_ids from January.
 If I just used 'append', I'd corrupt the downstream revenue reports.
 By using a 'left_anti' join, I’m effectively filtering out existing records,
 which makes this step idempotent—I can re-run this cell safely without 
 worrying about inflating the row count to 42.

In [0]:
feb_file = "orders_2024_02.csv"

feb_raw = (
    spark.read.csv(f"{DATA}/batch_incremental/{feb_file}", header=True, schema=orders_schema)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", lit(feb_file))
)

existing_ids = spark.read.format("delta").load(BRONZE_ORDERS_PATH).select("order_id")

# left_anti keeps only feb_raw rows whose order_id has NO match in existing_ids
feb_new = feb_raw.join(existing_ids, on="order_id", how="left_anti")

already_present = feb_raw.count() - feb_new.count()
print(f"Feb file rows: {feb_raw.count()} | already in bronze_orders: {already_present} | new rows to append: {feb_new.count()}")
if already_present > 0:
    dup_ids = [r["order_id"] for r in feb_raw.join(existing_ids, on="order_id", how="left_semi").select("order_id").collect()]
    print(f"  duplicate order_ids skipped: {dup_ids}")

feb_new.write.format("delta").mode("append").save(BRONZE_ORDERS_PATH)

total_after_feb = spark.read.format("delta").load(BRONZE_ORDERS_PATH).count()
print(f"bronze_orders after incremental load: {total_after_feb} rows")
assert total_after_feb == 40, f"expected 40 rows after Task 2, got {total_after_feb}"

Feb file rows: 22 | already in bronze_orders: 2 | new rows to append: 20
  duplicate order_ids skipped: ['ORD003', 'ORD013']
bronze_orders after incremental load: 40 rows


###Task 3 — Auto Loader: Streaming Transaction Ingestion 
Instead of manual batching, I’m using cloudFiles to watch the landing folder.
 This is much more scalable for production. I've set up a checkpoint directory
 so that the stream remembers its progress; if the job crashes, it won't 
 re-process the 15 transactions I've already saved to the Delta table.

In [0]:
AUTOLOADER_LANDING = (
    "/Volumes/databricksmaster/default/retailstream/data/autoloader_landing"
)
AUTOLOADER_CHECKPOINT = (
    "/Volumes/databricksmaster/default/retailstream/checkpoints/bronze_transactions"
)
AUTOLOADER_SCHEMA_LOC = "/Volumes/databricksmaster/default/retailstream/checkpoints/bronze_transactions_schema"
BRONZE_TXN_PATH = (
    "/Volumes/databricksmaster/default/retailstream/delta/bronze/bronze_transactions"
)

txn_stream_df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", AUTOLOADER_SCHEMA_LOC)
    .option("header", "true")
    .schema(transactions_schema)
    .load(AUTOLOADER_LANDING)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_name"))
)

query = (
    txn_stream_df.writeStream.format("delta")
    .option("checkpointLocation", AUTOLOADER_CHECKPOINT)
    .outputMode("append")
    .trigger(availableNow=True)
    .start(BRONZE_TXN_PATH)
)

query.awaitTermination()

bronze_txn = spark.read.format("delta").load(BRONZE_TXN_PATH)
txn_count = bronze_txn.count()
print(f"bronze_transactions after Auto Loader run: {txn_count} rows")
assert txn_count == 15, f"expected 15 rows after Task 3, got {txn_count}"

bronze_txn.orderBy("txn_id").show(5)

bronze_transactions after Auto Loader run: 15 rows
+------+--------+--------------+--------+-------------------+--------+--------------+--------------------+--------------------+
|txn_id|order_id|payment_method|  amount|      txn_timestamp|currency|gateway_status|         ingested_at|         source_file|
+------+--------+--------------+--------+-------------------+--------+--------------+--------------------+--------------------+
|TXN001| ORD_L04|           UPI| 2609.35|2024-03-01 03:00:00|     INR|        FAILED|2026-07-11 17:01:...|transactions_2024...|
|TXN002|  ORD030|           UPI|34412.27|2024-03-01 10:00:00|     INR|        FAILED|2026-07-11 17:01:...|transactions_2024...|
|TXN003|  ORD006|   NET_BANKING|16489.06|2024-03-02 03:00:00|     INR|       SUCCESS|2026-07-11 17:01:...|transactions_2024...|
|TXN004|  ORD031|   NET_BANKING|24321.13|2024-03-01 04:00:00|     INR|       SUCCESS|2026-07-11 17:01:...|transactions_2024...|
|TXN005|  ORD015|           UPI| 4806.61|2024-03-02 1

### Task 4 — Late Arriving File Handling (MERGE)
Since the S04 data for Jan arrived late, I’m using Delta's 'MERGE' command.
 A simple append would be risky if this file gets dropped again during a retry.
 'MERGE' checks for existing order_ids; if they exist, it skips them, 
and if they don't, it inserts them. This keeps the final row count at 45, 
ensuring the database state stays consistent.

In [0]:
late_file = "orders_2024_01_LATE.csv"

late_df = (
    spark.read.csv(f"{DATA}/late_arriving/{late_file}", header=True, schema=orders_schema)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", lit(late_file))
)

bronze_orders_delta = DeltaTable.forPath(spark, BRONZE_ORDERS_PATH)

(bronze_orders_delta.alias("target")
    .merge(late_df.alias("source"), "target.order_id = source.order_id")
    .whenNotMatchedInsertAll()
    .execute())

bronze_orders = spark.read.format("delta").load(BRONZE_ORDERS_PATH)
total_after_merge = bronze_orders.count()

# "records inserted by the merge" = rows whose source_file is the late file 
# NOT the same thing as "all S04 rows", since S04 also ships Feb orders
inserted_by_merge = bronze_orders.filter(col("source_file") == lit(late_file)).count()
s04_total = bronze_orders.filter(col("store_id") == "S04").count()

print(f"bronze_orders after MERGE: {total_after_merge} rows")
print(f"records inserted by this MERGE (source_file = '{late_file}'): {inserted_by_merge}")
print(f"store_id = 'S04' rows overall (Feb + late, for context): {s04_total}")
assert total_after_merge == 45, f"expected 45 rows after Task 4, got {total_after_merge}"
assert inserted_by_merge == 5, f"expected 5 records inserted by the late-file merge, got {inserted_by_merge}"

bronze_orders after MERGE: 45 rows
records inserted by this MERGE (source_file = 'orders_2024_01_LATE.csv'): 5
store_id = 'S04' rows overall (Feb + late, for context): 8


In [0]:
# Idempotency check : 
# same MERGE again and confirm the count does NOT change (should stay 45,not become 50). 
(bronze_orders_delta.alias("target")
    .merge(late_df.alias("source"), "target.order_id = source.order_id")
    .whenNotMatchedInsertAll()
    .execute())

recheck_count = spark.read.format("delta").load(BRONZE_ORDERS_PATH).count()
print(f"bronze_orders after re-running the MERGE: {recheck_count} rows ")
assert recheck_count == 45, "MERGE is not idempotent"

bronze_orders after re-running the MERGE: 45 rows 


### Task 5 — Silver Layer: Cleaned & Enriched Orders 
 This is where I clean and prepare the data for analytics. I’m joining the 
 fact table with products, customers, and stores. I’m using 'left joins' to 
 ensure I don't lose any orders if a foreign key is missing. After calculating 
 revenue and margins, I’m dropping the raw technical keys to keep the 
 Silver table clean and easy for the business team to read.

In [0]:
SILVER_ORDERS_PATH = "/Volumes/databricksmaster/default/retailstream/delta/silver/silver_orders"

bronze_orders = spark.read.format("delta").load(BRONZE_ORDERS_PATH)

silver_orders = (
    bronze_orders
    .join(products_df.select("product_id", "product_name", "category", "cost_price"), on="product_id", how="left")
    .join(customers_df.select("customer_id", "customer_name", "city", "tier"), on="customer_id", how="left")
    .join(stores_df.select("store_id", "store_name", "region"), on="store_id", how="left")
    .withColumn("revenue", _round(col("quantity") * col("unit_price"), 2))
    .withColumn("margin", _round(col("revenue") - (col("quantity") * col("cost_price")), 2))
    .drop("customer_id", "product_id", "store_id")
)

silver_orders.write.format("delta").mode("overwrite").save(SILVER_ORDERS_PATH)

silver_check = spark.read.format("delta").load(SILVER_ORDERS_PATH)
silver_count = silver_check.count()
print(f"silver_orders row count: {silver_count}")
assert silver_count == 45, f"expected 45 rows in silver_orders (all left joins on clean FKs), got {silver_count}"

silver_check.select("order_id", "product_name", "customer_name", "store_name", "revenue", "margin").show(5)

silver_orders row count: 45
+--------+------------+-------------+--------------+--------+--------+
|order_id|product_name|customer_name|    store_name| revenue|  margin|
+--------+------------+-------------+--------------+--------+--------+
|  ORD001|  Smartwatch|  Customer_26| Bangalore Hub| 28000.0| 25000.0|
|  ORD002|     Monitor|  Customer_21|     Delhi NCR| 72000.0| 40000.0|
|  ORD003|  Headphones|  Customer_20|Mumbai Central|100000.0| 97000.0|
|  ORD004|      Tablet|  Customer_31|     Delhi NCR| 18000.0|  3000.0|
|  ORD005|     Monitor|   Customer_5|     Delhi NCR|  2400.0|-21600.0|
+--------+------------+-------------+--------------+--------+--------+
only showing top 5 rows


### Task 6 — Gold Layer: Analytics with SparkSQL


 For the final layer, I’m using SparkSQL to answer specific business questions. 
 I’ve registered a temp view on the Silver data. This is great for readability—
 I can write standard SQL queries instead of complex PySpark dataframe transformations. 
 I’ve split 6a and 6b into two cells so it’s easy to re-run and verify the 
 sales trends and payment metrics independently.

### 6a. Monthly Sales Summary

In [0]:
GOLD_MONTHLY_SALES_PATH = "/Volumes/databricksmaster/default/retailstream/delta/gold/gold_monthly_sales"

silver_orders_final = spark.read.format("delta").load(SILVER_ORDERS_PATH)
silver_orders_final.createOrReplaceTempView("silver_orders")

monthly_sales = spark.sql('''
    SELECT
        DATE_FORMAT(order_date, 'yyyy-MM') AS month,
        COUNT(*)                            AS total_orders,
        ROUND(SUM(revenue), 2)              AS total_revenue,
        ROUND(SUM(margin), 2)               AS total_margin,
        ROUND(AVG(revenue), 2)              AS avg_order_value
    FROM silver_orders
    GROUP BY 1
    ORDER BY 1
''')

monthly_sales.write.format("delta").mode("overwrite").save(GOLD_MONTHLY_SALES_PATH)
monthly_sales.show()

+-------+------------+-------------+------------+---------------+
|  month|total_orders|total_revenue|total_margin|avg_order_value|
+-------+------------+-------------+------------+---------------+
|2024-01|          25|     731900.0|     71700.0|        29276.0|
|2024-02|          20|     959000.0|    615800.0|        47950.0|
+-------+------------+-------------+------------+---------------+



### 6b. Payment Method Summary

Pulling straight from `bronze_transactions` rather than Silver — payment gateway data doesn't need the
dimension enrichment Silver adds for orders, so promoting it through Silver first would just be an unused
extra hop.

In [0]:
GOLD_PAYMENT_SUMMARY_PATH = "/Volumes/databricksmaster/default/retailstream/delta/gold/gold_payment_summary"

bronze_transactions_final = spark.read.format("delta").load(BRONZE_TXN_PATH)
bronze_transactions_final.createOrReplaceTempView("bronze_transactions")

payment_summary = spark.sql('''
    SELECT
        payment_method,
        COUNT(*) AS total_transactions,
        ROUND(100.0 * SUM(CASE WHEN gateway_status = 'SUCCESS' THEN 1 ELSE 0 END) / COUNT(*), 2) AS success_rate,
        ROUND(SUM(amount), 2) AS total_amount
    FROM bronze_transactions
    GROUP BY payment_method
    ORDER BY total_transactions DESC
''')

payment_summary.write.format("delta").mode("overwrite").save(GOLD_PAYMENT_SUMMARY_PATH)
payment_summary.show()

+--------------+------------------+------------+------------+
|payment_method|total_transactions|success_rate|total_amount|
+--------------+------------------+------------+------------+
|           UPI|                 5|       60.00|    90437.22|
|    DEBIT_CARD|                 4|       75.00|   112412.42|
|   NET_BANKING|                 3|       66.67|    51857.39|
|   CREDIT_CARD|                 3|      100.00|   107487.12|
+--------------+------------------+------------+------------+



## Check — Delta time travel
 Verifying the pipeline's history using Time Travel,
 Checking Version 0 ensures the initial January load (20 rows) was successful,
 before any incremental or merge operations occurred.

In [0]:
v0_count = spark.read.format("delta").option("versionAsOf", 0).load(BRONZE_ORDERS_PATH).count()
print(f"bronze_orders @ version 0: {v0_count} rows (expected 20 - the Jan-only initial load)")
assert v0_count == 20

spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_ORDERS_PATH}`").select("version", "timestamp", "operation").show(truncate=False)

bronze_orders @ version 0: 20 rows (expected 20 - the Jan-only initial load)
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|10     |2026-07-11 20:55:10|MERGE    |
|9      |2026-07-11 20:55:07|MERGE    |
|8      |2026-07-11 20:54:59|WRITE    |
|7      |2026-07-11 20:54:55|WRITE    |
|6      |2026-07-11 20:51:16|WRITE    |
|5      |2026-07-11 17:20:30|MERGE    |
|4      |2026-07-11 17:19:44|MERGE    |
|3      |2026-07-11 17:15:32|MERGE    |
|2      |2026-07-11 16:57:39|WRITE    |
|1      |2026-07-11 16:54:26|WRITE    |
|0      |2026-07-11 16:49:07|WRITE    |
+-------+-------------------+---------+

